# 🥈 Silver Layer: Clean and Deduplicate
**Purpose:** Read the raw data from Bronze, cast data types (like Date), remove invalid records, and UPSERT (Merge) into the Silver table to prevent duplicates if the pipeline runs twice in one day.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# 1. Configuration
CATALOG = "portfolio"
SCHEMA = "market_data"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_stock_quotes"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_stock_quotes"

In [0]:
# 2. Read and Cleanse
df_raw = spark.read.table(BRONZE_TABLE)

df_silver = df_raw.withColumn("Date", F.to_date(F.col("Date"), "yyyy-MM-dd")) \
                  .withColumnRenamed("Adj Close", "Adj_Close") \
                  .dropDuplicates(["ticker_symbol", "Date"]) \
                  .dropna(subset=["Close", "Date"]) # Drop rows missing critical data

In [0]:
# 3. Upsert into Silver
if spark.catalog.tableExists(SILVER_TABLE):
    delta_table = DeltaTable.forName(spark, SILVER_TABLE)
    
    # Merge based on Date AND Ticker (future-proofing for multiple stocks)
    delta_table.alias("target").merge(
        df_silver.alias("source"),
        "target.Date = source.Date AND target.ticker_symbol = source.ticker_symbol"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    # Initial creation
    df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

display(spark.read.table(SILVER_TABLE).orderBy(F.col("Date").desc()).limit(5))